In [7]:


from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
import os

from openai.types import upload

load_dotenv()
api_key = os.getenv("DASHSCOPE_API_KEY")

model = init_chat_model(
    model="qwen-plus",
    model_provider="openai",
    api_key=api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    temperature=0,
)
agent = create_agent(model)
response = agent.invoke({
    "messages": [
        HumanMessage(content="你好，我是cc"),
        AIMessage(content="你好，cc，很高兴认识你。"),
        HumanMessage(content="我的名字是什么？")
    ]
})
print(response)

{'messages': [HumanMessage(content='你好，我是cc', additional_kwargs={}, response_metadata={}, id='ba19b310-8781-4640-a6b2-3cbe00cad4cc'), AIMessage(content='你好，cc，很高兴认识你。', additional_kwargs={}, response_metadata={}, id='136d9e21-b988-4950-9296-c311f5c8572e', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我的名字是什么？', additional_kwargs={}, response_metadata={}, id='f88ca231-7c47-4e16-a2d8-0b7915b8a905'), AIMessage(content='你的名字是 **cc** 😊  \n（简洁有力，还带点酷酷的感觉～）  \n需要我帮你做点什么，或者想聊些什么？欢迎随时告诉我！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 34, 'total_tokens': 72, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'qwen-plus', 'system_fingerprint': None, 'id': 'chatcmpl-837680ba-eba1-9c69-b194-f57dfd18ab3a', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e6a6e-2465-7bd3-b053-54b32611f9aa-0', tool_calls=[], inval

In [4]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

你好，我是cc
================================== Ai Message ==================================

你好，cc，很高兴认识你。
================================ Human Message =================================

我的名字是什么？
================================== Ai Message ==================================

你的名字是 **cc** 😊  
（如果你有更正式的名字、昵称，或者希望我用特定方式称呼你，也欢迎告诉我～）


多模态

In [13]:
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://my.feishu.cn/04b26100-559f-4eb1-9f6c-d5c8beaf09ec"},
        {"type": "text", "text": "这些图描绘了什么内容？"}
    ])


for token, metadata in agent.stream({
    "messages": [multimodal_message]
}, stream_mode="messages"):
    if token.content:
        print(token.content, end="", flush=True)


您好！您提到了“这些图”，但当前对话中并未附带任何图片或图像文件。我无法查看、分析或描述未提供的图像内容。

如果您希望了解某张图（或几幅图）所描绘的内容，请您：

✅ 上传图片（如支持的格式：PNG、JPG、JPEG等）；  
✅ 或提供详细的文字描述（例如：图中有哪些元素？颜色、文字、图表类型、场景等）；  
✅ 或说明图像来源（如来自某篇论文、教材、网页截图等），我可以尝试结合上下文帮助解读。

期待您的补充信息，我很乐意为您详细分析！ 📊🔍

In [14]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept="image/*",multipart=False)
display(uploader)


FileUpload(value=(), accept='image/*', description='Upload')

In [15]:
import base64
uploaded_file = uploader.value[0]
content = uploaded_file["content"]
img_bytes = bytes(content)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

# 调用Agent，发送消息
response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

您好！不过目前我无法查看或分析图片——作为文字型AI，我没有图像识别或访问您设备中图片的能力。如果您能提供以下任意信息，我很乐意帮您详细介绍该城市：

✅ 城市名称（例如：西安、伊斯坦布尔、京都）  
✅ 图片中的显著特征（如：金色穹顶的教堂、蜿蜒的运河、高耸的玻璃塔楼、古城墙与现代高楼并存等）  
✅ 地理线索（如：位于地中海沿岸、地处高原盆地、有著名火山背景等）  
✅ 文化或历史元素（如：有兵马俑雕塑、清真寺宣礼塔、樱花大道、涂鸦艺术街区等）

只要您描述一下，我就能为您介绍它的历史、文化、地标、特色美食、旅行贴士，甚至冷知识！😊  
期待您的补充～ 🌍
